# 3. The agent loop, by hand

**LangGraph tutorial, lesson 3 of 5**

Lesson 2 did one round trip. Real questions need an unknown number, so the round
trip has to become a loop:

```
while the model keeps asking for tools:
    run them, hand back the results, ask again
```

**That loop is the agent.** Fifteen lines, no framework. Write it once by hand
and LangGraph stops looking like magic — you will recognise it as this loop with
better plumbing.

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

from common import make_llm, text_of
from tools import ALL_TOOLS

llm_with_tools = make_llm().bind_tools(ALL_TOOLS)
tools_by_name = {t.name: t for t in ALL_TOOLS}

# A guard rail. Without it, a confused model that keeps requesting tools would
# loop until your budget runs out. ALWAYS bound an agent loop.
MAX_ITERATIONS = 8

## A question that needs several tools

This one cannot be answered in a single pass. The model must:

1. look up Katherine Johnson → discover her manager is Ada Lovelace
2. look up *Ada Lovelace* → get her start year
3. check the current date
4. subtract

Step 2 is impossible until step 1 comes back. **The model has to sequence these
itself** — we never tell it the order.

In [ ]:
QUESTION = (
    "Who is Katherine Johnson's manager, and how many years has that manager "
    "worked here as of today?"
)

messages = [
    SystemMessage(
        "You are a helpful assistant. Use the provided tools rather than "
        "guessing. Never do arithmetic yourself -- always use the calculator."
    ),
    HumanMessage(QUESTION),
]

## The loop

In [ ]:
for iteration in range(1, MAX_ITERATIONS + 1):
    print(f"--- iteration {iteration}: calling the model ---")

    reply = llm_with_tools.invoke(messages)
    messages.append(reply)                    # remember what it said

    said = text_of(reply)
    if said:
        print(f"  model says: {said[:120]}")

    # THE EXIT CONDITION. No tool calls means the model is done asking for
    # things and has produced its final answer.
    if not reply.tool_calls:
        print("  no tool calls -> final answer")
        break

    # Otherwise: run everything it asked for, append each result.
    for request in reply.tool_calls:
        tool = tools_by_name.get(request["name"])
        if tool is None:
            # Hallucinated tool name. Tell the model instead of crashing --
            # it can usually recover and pick a real one.
            result = f"Error: no such tool {request['name']!r}."
        else:
            result = tool.invoke(request["args"])

        print(f"  -> {request['name']}({request['args']})")
        print(f"     = {str(result).replace(chr(10), ' | ')[:90]}")

        messages.append(ToolMessage(content=str(result), tool_call_id=request["id"]))
else:
    print(f"Gave up after {MAX_ITERATIONS} iterations.")

In [ ]:
print("FINAL ANSWER\n")
print(text_of(messages[-1]))
print(f"\nModel calls: {iteration}   Messages held: {len(messages)}")

## What this cost

The message list only grows. That is the agent's entire memory, and it is
**re-sent in full on every iteration** — which is why long agent runs get
expensive, and why the context window is the real constraint.

## So why bother with LangGraph?

The loop above works. For a notebook this size it is genuinely fine.

Now try to add what production needs:

- stream progress to a UI as it happens
- **pause for human approval** before a risky tool runs
- resume after a crash without replaying everything
- branch to different tool sets depending on the question
- trace every step for debugging

Each one means threading more state through that `for` loop, and they compose
badly. LangGraph models the loop as a **graph** — nodes and edges — so these
features attach to the structure instead of tangling your control flow.

Being fair about the trade: a `while` loop you can read top to bottom becomes a
set of nodes whose execution order is inferred. For five tools and a UI that is
a clear win. For one tool and no UI, the loop may be the better code.

---
**Next:** `04_first_graph.ipynb`